In [4]:
# TUGAS AKHIR PENGOLAHAN CITRA DIGITAL
# PEMAMPATAN CITRA MENGGUNAKAN RUN-LENGTH ENABLING (RLE)
# Rahmat Eka Satria - 231011402890
# Universitas Pamulang - Semester 5

import numpy as np
from PIL import Image
import io
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript
import base64
import matplotlib.pyplot as plt

# Font profesional
display(widgets.HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
.widget-area * { font-family: 'Inter', sans-serif !important; }
</style>
"""))

# ==================== RLE & UTILITIES ====================
def rle_encode(data):
    encoded = []
    i = 0
    while i < len(data):
        val = data[i]
        count = 1
        i += 1
        while i < len(data) and data[i] == val:
            count += 1
            i += 1
        encoded.append((val, count))
    return encoded

def rle_decode(encoded, h, w):
    return np.array([v for v, c in encoded for _ in range(c)], dtype=np.uint8).reshape(h, w)

def create_histogram(img_array, title):
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    colors = ['red', 'green', 'blue']
    for i, col in enumerate(colors):
        ax[i].hist(img_array[:, :, i].ravel(), bins=256, range=(0, 255), color=col, alpha=0.75, edgecolor='black', linewidth=0.5)
        ax[i].set_title(f'{col.upper()}', fontsize=12, fontweight='bold')
        ax[i].set_xlim(0, 255)
        ax[i].grid(True, alpha=0.3)
    plt.suptitle(title, fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=180, bbox_inches='tight', facecolor='#f8fafc')
    plt.close(fig)
    return buf.getvalue()

# ==================== UI ====================
header = widgets.HTML(f"""
<div style="background:linear-gradient(135deg, #1e40af, #3b82f6); color:white; padding:30px; border-radius:22px; text-align:center;
            box-shadow: 0 14px 42px rgba(30,100,200,0.3); margin-bottom:30px;">
    <h1 style="margin:0; font-size:40px; font-weight:700;">RLE Image Compression</h1>
    <p style="margin:10px 0 0; font-size:19px;">Run-Length Encoding • Lossless • Analisis Lengkap</p>
</div>
<div style="text-align:center; background:#f0f9ff; padding:18px; border-radius:16px; 
            box-shadow: 0 10px 30px rgba(0,0,0,0.1); margin-bottom:35px;">
    <h3 style="margin:0; color:#1e40af;">Rahmat Eka Satria</h3>
    <p style="margin:6px 0 0; color:#475569; font-weight:500;">231011402890 • Universitas Pamulang • Semester 5</p>
</div>
""")

uploader = widgets.FileUpload(accept='image/*', multiple=False, description='Pilih Gambar',
                              icon='image', layout=widgets.Layout(width='270px', height='58px'))

btn_proses = widgets.Button(description='Proses RLE', icon='play-circle',
                            layout=widgets.Layout(width='270px', height='58px', margin='0 15px'))
btn_proses.style.button_color = '#059669'
btn_proses.style.font_weight = '600'

btn_download = widgets.Button(description='Download Hasil RLE (.txt)', icon='download',
                              layout=widgets.Layout(width='360px', height='54px', margin='25px auto'), disabled=True)
btn_download.style.button_color = '#dc2626'
btn_download.style.font_weight = '600'

img_container = widgets.HBox([], layout=widgets.Layout(justify_content='center', gap='50px', margin='40px 0'))
histogram_box = widgets.HBox([], layout=widgets.Layout(justify_content='center', gap='50px', margin='35px 0'))
info_box = widgets.HTML()
result_box = widgets.HTML()
out = widgets.Output()

# Global
rle_data_global = None
filename_global = "hasil_rle.txt"

# ==================== DOWNLOAD ====================
def download_rle(b):
    global rle_data_global, filename_global
    if not rle_data_global: return
    
    text = f"""PEMAMPATAN CITRA MENGGUNAKAN RUN-LENGTH ENCODING (RLE)
Nama       : Rahmat Eka Satria
NIM        : 231011402890
Mata Kuliah: Pengolahan Citra Digital
Semester   : 5
File       : {filename_global.replace('.txt', '')}
{'='*80}

HASIL RLE LENGKAP PER CHANNEL:
"""
    for nama, data in zip(["RED", "GREEN", "BLUE"], rle_data_global):
        runs = " ".join(f"({v},{c})" for v,c in data)
        text += f"\n{nama}:\n{runs}\n"
    
    b64 = base64.b64encode(text.encode()).decode()
    display(Javascript(f"""
    var link = document.createElement('a');
    link.href = 'data:text/plain;base64,{b64}';
    link.download = '{filename_global}';
    document.body.appendChild(link);
    link.click();
    document.body.removeChild(link);
    """))

btn_download.on_click(download_rle)

# ==================== PROSES UTAMA ====================
def proses(b):
    global rle_data_global, filename_global
    with out:
        clear_output()
        if not uploader.value:
            info_box.value = '<div style="padding:26px; background:#fee2e2; color:#991b1b; border-radius:16px; text-align:center; box-shadow:0 10px 30px rgba(0,0,0,0.1);"><h3>Upload gambar terlebih dahulu!</h3></div>'
            return

        file = uploader.value[0]
        nama_file = getattr(file, 'name', 'gambar')
        filename_global = f"RLE_{nama_file.split('.')[0] if '.' in nama_file else nama_file}.txt"

        img = Image.open(io.BytesIO(file.content.tobytes())).convert('RGB')
        arr = np.array(img)
        h, w = arr.shape[:2]
        total_pixels = h * w

        # === UKURAN SEBELUM KOMPRESI ===
        bits_per_pixel = 24  # RGB 8-bit each
        original_bits = total_pixels * bits_per_pixel
        original_bytes = original_bits // 8

        # === RLE ===
        enc_r = rle_encode(arr[:,:,0].ravel())
        enc_g = rle_encode(arr[:,:,1].ravel())
        enc_b = rle_encode(arr[:,:,2].ravel())
        rle_data_global = [enc_r, enc_g, enc_b]
        total_runs = sum(len(ch) for ch in rle_data_global)

        # === UKURAN SESUDAH RLE (estimasi real) ===
        # Setiap run: 8 bit (value) + 32 bit (count, karena count bisa besar) = 40 bit/run
        bits_per_run = 40
        rle_bits = total_runs * bits_per_run
        rle_bytes = rle_bits // 8

        # Rasio kompresi
        ratio = round(original_bits / rle_bits, 2) if rle_bits > 0 else 0
        saving_percent = round((1 - rle_bits / original_bits) * 100, 2)

        # Rekonstruksi
        recon = np.stack([rle_decode(ch, h, w) for ch in rle_data_global], axis=2)
        recon_img = Image.fromarray(recon)

        # Gambar card
        def card(pil_img, title):
            buf = io.BytesIO(); pil_img.save(buf, 'PNG')
            return widgets.VBox([
                widgets.HTML(f"<h3 style='text-align:center; margin:12px 0; color:#1e293b;'>{title}</h3>"),
                widgets.Image(value=buf.getvalue(), format='png', width=440, height=440)
            ], layout=widgets.Layout(
                padding='24px', background='white', border_radius='22px',
                box_shadow='0 16px 45px rgba(0,0,0,0.15)', border='1px solid #e2e8f0'
            ))

        img_container.children = [card(img, "Gambar Asli"), card(recon_img, "Hasil Rekonstruksi")]

        # Histogram
        hist_before = create_histogram(arr, "Histogram Gambar Asli")
        hist_after = create_histogram(recon, "Histogram Hasil Rekonstruksi")
        histogram_box.children = [
            widgets.Image(value=hist_before, width=620),
            widgets.Image(value=hist_after, width=620)
        ]

        # Info box — LENGKAP SEBELUM & SESUDAH
        info_box.value = f"""
        <div style="background:white; padding:32px; border-radius:20px; text-align:center;
                    box-shadow:0 16px 45px rgba(0,0,0,0.13); border:1px solid #e2e8f0;">
            <h2 style="color:#1e293b; margin-top:0;">Analisis Lengkap Pemampatan RLE</h2>
            <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:28px; margin:32px 0; font-size:17px;">
                <div><b style="color:#64748b;">Total Pixel</b><br><span style="font-size:28px; color:#1d4ed8;">{total_pixels:,}</span></div>
                <div><b style="color:#64748b;">Bit per Pixel</b><br><span style="font-size:28px; color:#7c2d12;">{bits_per_pixel} bit</span></div>
                <div><b style="color:#dc2626;">Sebelum Kompresi</b><br><span style="font-size:28px;">{original_bytes:,} bytes</span><br><small style="color:#64748b;">({original_bits:,} bit)</small></div>
                <div><b style="color:#059669;">Sesudah RLE</b><br><span style="font-size:28px;">{rle_bytes:,} bytes</span><br><small style="color:#64748b;">({rle_bits:,} bit)</small></div>
                <div><b style="color:#64748b;">Jumlah Run</b><br><span style="font-size:28px; color:#1d4ed8;">{total_runs:,}</span></div>
                <div><b style="color:#7c2d12;">Rasio Kompresi</b><br><span style="font-size:38px; font-weight:700;">{ratio}×</span></div>
                <div><b style="color:#059669;">Penghematan</b><br><span style="font-size:32px; font-weight:700;">{saving_percent}%</span></div>
            </div>
            <p style="color:#059669; font-weight:bold; font-size:18px; margin:25px 0 0;">
                Gambar 100% identik • Lossless • Histogram sama persis
            </p>
        </div>
        """

        # Hasil RLE
        sample = 60
        runs = ""
        for nama, warna, data in zip(["RED", "GREEN", "BLUE"], ["#ef4444", "#10b981", "#3b82f6"], rle_data_global):
            s = "  ".join(f"<code style='background:{warna}; color:white; padding:3px 8px; border-radius:6px;'>({v},{c})</code>" for v,c in data[:sample])
            if len(data) > sample: s += " ..."
            runs += f"<div style='margin:20px 0; padding:20px; background:#f8fafc; border-left:7px solid {warna}; border-radius:14px; font-family:monospace;'>{nama} → {s}</div>"

        result_box.value = f"""
        <div style="background:white; padding:34px; border-radius:20px; margin-top:35px;
                    box-shadow:0 16px 45px rgba(0,0,0,0.13); border:1px solid #e2e8f0;">
            <h3 style="text-align:center; color:#1e293b; margin-top:0;">Hasil Run-Length Encoding</h3>
            {runs}
            <p style="text-align:center; color:#64748b; margin-top:28px; font-size:15px;">
                Menampilkan {sample} run pertama per channel • Total run: {total_runs:,}
            </p>
        </div>
        """

        btn_download.disabled = False
        btn_download.description = f"Download {filename_global}"

btn_proses.on_click(proses)

# ==================== TAMPILAN AKHIR ====================
display(widgets.VBox([
    header,
    widgets.HBox([uploader, btn_proses], layout=widgets.Layout(justify_content='center', margin='35px 0')),
    out,
    img_container,
    info_box,
    widgets.HTML("<h3 style='text-align:center; color:#1e293b; margin:40px 0 15px; font-weight:600;'>Histogram RGB (Sebelum vs Sesudah)</h3>"),
    histogram_box,
    btn_download,
    result_box
], layout=widgets.Layout(padding='35px', background='#f1f5f9', align_items='center')))

HTML(value="\n<style>\n@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&displa…